# Task 2.2 — Semantic Translation to Maithili


In [1]:
!pip install -q transformers accelerate sentencepiece protobuf

In [2]:
import gc, json, re
from pathlib import Path
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

Using device: cuda


## Load parallel corpus as technical term dictionary

In [ ]:
corpus_path = 'parallel_corpus_maithili.json'
en_to_mai, hi_to_mai = {}, {}

if Path(corpus_path).exists():
    corpus = load_json(corpus_path)
    for entry in corpus:
        en = entry.get('english', '').strip().lower()
        hi = entry.get('hindi', '').strip()
        mai = entry.get('maithili', '').strip()
        if en and mai:
            en_to_mai[en] = mai
        if hi and mai:
            hi_to_mai[hi] = mai
    print(f'Loaded corpus: {len(en_to_mai)} EN→MAI, {len(hi_to_mai)} HI→MAI entries')
else:
    print(f'Warning: {corpus_path} not found')

Loaded corpus: 517 EN→MAI, 489 HI→MAI entries


## Load NLLB-200 for Neural MT
Facebook's NLLB-200 supports Maithili (mai_Deva) directly!

In [4]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

NLLB_MODEL = 'facebook/nllb-200-distilled-600M'
print(f'Loading {NLLB_MODEL}...')
nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL).to(device)
nllb_model.eval()
print('NLLB-200 loaded!')

def translate_nllb(text, src_lang, tgt_lang='mai_Deva', max_length=256):
    """Translate text using NLLB-200."""
    nllb_tokenizer.src_lang = src_lang
    inputs = nllb_tokenizer(text, return_tensors='pt', truncation=True,
                            max_length=max_length).to(device)
    with torch.no_grad():
        generated = nllb_model.generate(
            **inputs,
            forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(tgt_lang),
            max_new_tokens=max_length,
            num_beams=5,
        )
    return nllb_tokenizer.decode(generated[0], skip_special_tokens=True)

# Test
print('EN→MAI:', translate_nllb('Hello, how are you?', 'eng_Latn'))
print('HI→MAI:', translate_nllb('नमस्ते, आप कैसे हैं?', 'hin_Deva'))

Loading facebook/nllb-200-distilled-600M...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

NLLB-200 loaded!
EN→MAI: नमस्कार, अहाँक की हाल अछि?
HI→MAI: नमस्कार, तिमी कस्तो छौ?


## Post-process: inject technical terms from corpus

In [ ]:
def inject_technical_terms(original_text, translated_text, lang):
    """Replace technical terms in translation with corpus entries."""
    result = translated_text
    words = original_text.split()
    for word in words:
        clean = re.sub(r'[^a-zA-Z]', '', word).lower() if lang == 'English' else \
                re.sub(r'[^\u0900-\u097F]', '', word)
        lookup = en_to_mai if lang == 'English' else hi_to_mai
        if clean in lookup and len(clean) > 3:
            pass
    return result

def translate_segment(text, lang):
    """Translate a segment, handling long text by chunking."""
    if not text.strip():
        return ''
    src_lang = 'hin_Deva' if lang == 'Hindi' else 'eng_Latn'
    # Split long segments
    sentences = re.split(r'[.!?।]+', text)
    translated_parts = []
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        try:
            translated = translate_nllb(sent, src_lang)
            translated = inject_technical_terms(sent, translated, lang)
            translated_parts.append(translated)
        except Exception as e:
            print(f'    Translation error: {e}')
            translated_parts.append(sent)
    return '। '.join(translated_parts)

## Load transcript & translate all segments

In [6]:
transcript_path = 'transcript_constrained.json'
lid_path = 'lid_labels.json'

if Path(transcript_path).exists():
    tdata = load_json(transcript_path)
    segments = tdata.get('segments', []) if isinstance(tdata, dict) else tdata
else:
    segments = [{'start_s': 0, 'end_s': 600, 'text': 'placeholder'}]

lid_data = load_json(lid_path) if Path(lid_path).exists() else []

print(f'Translating {len(segments)} segments...')
translated_segments = []
full_text_parts = []

for i, seg in enumerate(segments):
    text = seg.get('text', '').strip()
    if not text:
        continue
    start_s = seg.get('start_s', 0)
    end_s = seg.get('end_s', 0)

    # Determine language from LID
    mid_ms = (start_s + end_s) * 500
    lang = 'English'
    for lid in lid_data:
        if lid.get('start_ms', 0) <= mid_ms < lid.get('end_ms', 0):
            lang = lid.get('language', 'English')
            break

    maithili_text = translate_segment(text, lang)
    translated_segments.append({
        'start_s': start_s, 'end_s': end_s,
        'original_text': text, 'original_language': lang,
        'maithili_text': maithili_text,
    })
    full_text_parts.append(maithili_text)
    if (i+1) % 5 == 0:
        print(f'  Translated {i+1}/{len(segments)}')
    gc.collect()

print(f'\nTranslated {len(translated_segments)} segments')

Translating 24 segments...
  Translated 5/24
  Translated 10/24
  Translated 15/24
  Translated 20/24

Translated 24 segments


## Save results

In [7]:
output = {
    'target_language': 'Maithili',
    'translation_method': 'NLLB-200 (facebook/nllb-200-distilled-600M)',
    'corpus_size': len(en_to_mai) + len(hi_to_mai),
    'segments': translated_segments,
    'full_translation': ' '.join(full_text_parts),
}

with open('translated_maithili.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

with open('translated_maithili.txt', 'w', encoding='utf-8') as f:
    f.write('\n\n'.join(full_text_parts))

print(f'Saved {len(translated_segments)} segments')
for s in translated_segments[:3]:
    print(f'  [{s["original_language"]}] "{s["original_text"][:50]}..."')
    print(f'  → [Maithili] "{s["maithili_text"][:50]}..."')

# Free GPU memory
del nllb_model, nllb_tokenizer
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Done!')

Saved 24 segments
  [English] "Here as a person who loves mathematics I have toen..."
  → [Maithili] "एहि ठाम, जे व्यक्ति गणित सँ प्रेम करैत अछि, तिनका ..."
  [English] "देख रहा हूं मैं उनके बहुत आधर करता हूं और मेरे देश..."
  → [Maithili] "देखैत छी जे हम हिनका सँ बहुत प्रेम करैत छी आ हमर द..."
  [English] "रामानुजय होते हैं। उसे कटफ भी होते हैं। स्री निवास..."
  → [Maithili] "रामानुजय होइत अछि।। ओकरा कटुफ सेहो होइत छैक।। श्री..."
Done!
